# 🚀 pgVectorDB v0.0.6 — Unified API Quick Start

This notebook demonstrates the new **Unified Query API** introduced in pgVectorDB v0.0.6.

## What's New

- **Single entry point**: `db.query("...")` for all search methods
- **Fluent API**: Chain methods for clean, readable code
- **Search modes**: Semantic, Keyword (BM25/FTS), Hybrid, Trigram
- **Multimodal support**: Search across text, numbers, categories
- **Query analysis**: Built-in explain and analyze capabilities

### Prerequisites
- PostgreSQL with `pgvector` extension
- Python dependencies: `pip install pgvectordb[huggingface]`

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from langchain_core.documents import Document

from pgvectordb import Config, SearchMethod, pgVectorDB

print("✅ Imports successful")

✅ Imports successful


## 1. Initialize Database

Connect to PostgreSQL and create the vector store.

In [2]:
db = pgVectorDB(
    collection_name="nb_unified_api",
    embedding_model=Config.get_embeddings(),
    connection_string=Config.get_connection_string(),
)
await db.initialize(overwrite_existing=True)
print("✅ Database initialized")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Database initialized


## 2. Add Sample Documents

We'll add documents with rich metadata for filtering and multimodal search.

In [3]:
docs = [
    Document(
        page_content="PostgreSQL is a powerful open-source relational database system with advanced features.",
        metadata={"category": "database", "year": 2024, "priority": 9, "price": 0},
    ),
    Document(
        page_content="pgvector adds vector similarity search capabilities to PostgreSQL for AI applications.",
        metadata={"category": "database", "year": 2024, "priority": 10, "price": 0},
    ),
    Document(
        page_content="Machine learning models convert text into dense vector embeddings for semantic search.",
        metadata={"category": "ai", "year": 2023, "priority": 8, "price": 50000},
    ),
    Document(
        page_content="RAG combines retrieval and generation for accurate AI responses with sources.",
        metadata={"category": "ai", "year": 2024, "priority": 9, "price": 75000},
    ),
    Document(
        page_content="Docker containers package applications with all dependencies for consistent deployment.",
        metadata={"category": "devops", "year": 2022, "priority": 7, "price": 299},
    ),
    Document(
        page_content="Kubernetes orchestrates containerized workloads at scale across clusters.",
        metadata={"category": "devops", "year": 2023, "priority": 8, "price": 499},
    ),
]

ids = await db.add_documents(docs)
print(f"✅ Added {len(ids)} documents")

✅ Added 6 documents


## 3. Semantic Search (Default)

The default search mode uses vector similarity.

In [4]:
# Simple semantic search
results = await db.query("vector database AI").limit(3).to_list()

print("🔍 Semantic Search Results:")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:60]}...")

🔍 Semantic Search Results:
  1. [0.3794] pgvector adds vector similarity search capabilities to Postg...
  2. [0.5673] PostgreSQL is a powerful open-source relational database sys...
  3. [0.5962] Machine learning models convert text into dense vector embed...


## 4. Keyword Search with BM25

Use `.search_mode(SearchMethod.KEYWORD)` for BM25 ranking.

In [5]:
await db.build_bm25_index(k1=1.2, b=0.75)

results = await (
    db.query("database search")
    .search_mode(SearchMethod.KEYWORD)
    .bm25_params(k1=1.2, b=0.75)  # BM25 parameters
    .limit(3)
    .to_list()
)

print("🔤 BM25 Keyword Search:")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:60]}...")

🔤 BM25 Keyword Search:
  1. [1.4367] PostgreSQL is a powerful open-source relational database sys...
  2. [1.0054] pgvector adds vector similarity search capabilities to Postg...
  3. [0.9603] Machine learning models convert text into dense vector embed...


## 5. Hybrid Search (Vector + Keyword)

Combine semantic and keyword signals with weighted fusion or RRF.

## NEW: Metadata-Only Search

Use `.metadata_only()` to filter documents by metadata without any text search.
This is useful when you want to retrieve documents purely by their metadata fields.

In [ ]:
# Metadata-only search (no text query needed)
meta_results = await (
    db.query("")
    .metadata_only()
    .where({"category": "database"})
    .limit(5)
    .to_list()
)

print(f"Found {len(meta_results)} database documents (metadata only):")
for r in meta_results:
    print(f"  • {r['content'][:50]}...")

## NEW: Ensemble Search

Use `.ensemble()` for hybrid search on a filtered subset.
This is a convenience method that combines filtered semantic + keyword search.

In [ ]:
# Ensemble search: hybrid on filtered subset
ensemble_results = await (
    db.query("search optimization")
    .ensemble()
    .where({"category": "database"})
    .weights(semantic=0.6, keyword=0.4)
    .limit(5)
    .to_list()
)

print(f"Found {len(ensemble_results)} documents (ensemble search):")
for r in ensemble_results:
    print(f"  • [{r['score']:.4f}] {r['content'][:50]}...")

In [6]:
# Weighted fusion
results = await (
    db.query("vector database performance")
    .search_mode(SearchMethod.HYBRID)
    .weights(semantic=0.7, keyword=0.3)  # 70% semantic, 30% keyword
    .limit(3)
    .to_list()
)

print("⚡ Hybrid Search (Weighted):")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:60]}...")

⚡ Hybrid Search (Weighted):
  1. [0.8572] PostgreSQL is a powerful open-source relational database sys...
  2. [0.7284] pgvector adds vector similarity search capabilities to Postg...
  3. [0.3458] Machine learning models convert text into dense vector embed...


In [7]:
# RRF fusion (Reciprocal Rank Fusion)
results = await (
    db.query("vector database performance")
    .search_mode(SearchMethod.HYBRID)
    .rrf(k=60)  # RRF constant
    .limit(3)
    .to_list()
)

print("⚡ Hybrid Search (RRF):")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:60]}...")

⚡ Hybrid Search (RRF):
  1. [0.0325] pgvector adds vector similarity search capabilities to Postg...
  2. [0.0325] PostgreSQL is a powerful open-source relational database sys...
  3. [0.0317] Machine learning models convert text into dense vector embed...


## 8. Metadata Filtering

Use `.where()` for MongoDB-style filtering.

In [8]:
# Filter by category and year
results = await (
    db.query("AI systems").where({"category": "ai", "year": {"$gte": 2023}}).limit(5).to_list()
)

print("📋 Filtered Results (category=ai, year≥2023):")
for r in results:
    print(f"  • {r['content'][:50]}... ({r['metadata']})")

📋 Filtered Results (category=ai, year≥2023):
  • RAG combines retrieval and generation for accurate... ({'category': 'ai', 'year': 2024, 'priority': 9, 'price': 75000, 'langchain_id': 'dfde0e79-1b17-449b-ae8d-2812959d3b76'})
  • Machine learning models convert text into dense ve... ({'category': 'ai', 'year': 2023, 'priority': 8, 'price': 50000, 'langchain_id': '39e2ab8d-0871-4827-884f-5c9db1aa8d27'})


In [9]:
# Complex filter with $and/$or
results = await (
    db.query("database")
    .where({"$or": [{"category": "database"}, {"priority": {"$gte": 9}}]})
    .limit(5)
    .to_list()
)

print("📋 Complex Filter Results:")
for r in results:
    print(f"  • {r['content'][:50]}... (priority: {r['metadata'].get('priority')})")

📋 Complex Filter Results:
  • PostgreSQL is a powerful open-source relational da... (priority: 9)
  • pgvector adds vector similarity search capabilitie... (priority: 10)
  • RAG combines retrieval and generation for accurate... (priority: 9)


## 9. Query Parameter Tuning

Fine-tune vector search parameters.

In [10]:
# HNSW parameter tuning
results = await (
    db.query("vector embeddings")
    .ef(100)  # Increase candidate pool for better recall
    .refine_factor(2)  # Oversample and rerank
    .limit(3)
    .to_list()
)

print("🎛️ Tuned Search Results:")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content'][:50]}...")

🎛️ Tuned Search Results:
  1. [0.5299] Machine learning models convert text into dense ve...
  2. [0.6508] pgvector adds vector similarity search capabilitie...
  3. [0.8608] RAG combines retrieval and generation for accurate...


## 10. Query Analysis

Analyze query execution with `.explain_plan()` and `.analyze_plan()`.

In [11]:
# Explain plan (no execution)
plan = db.query("test").where({"category": "ai"}).explain_plan()

print("📊 Query Plan:")
for key, value in plan.items():
    print(f"  {key}: {value}")

📊 Query Plan:
  search_method: semantic
  query: test
  filter: {'category': 'ai'}
  limit: 10
  index_type: hnsw


In [12]:
# Analyze with execution metrics
metrics = await db.query("vector database").where({"category": "database"}).limit(5).analyze_plan()

print("📈 Execution Metrics:")
for key, value in metrics.items():
    if key != "config":  # Skip verbose config
        print(f"  {key}: {value}")

📈 Execution Metrics:
  execution_time_ms: 12.141227722167969
  rows_returned: 2
  search_method: semantic


## 11. Output Formats

Get results in different formats.

In [13]:
# As pandas DataFrame
df = await db.query("test").limit(3).to_pandas()
print("📊 Pandas DataFrame:")
print(df[["content", "score"]].head())

📊 Pandas DataFrame:


                                             content     score
0  RAG combines retrieval and generation for accu...  0.800537
1  Machine learning models convert text into dens...  0.928721
2  PostgreSQL is a powerful open-source relationa...  0.953155


## 12. Comparison: Old vs New API

The legacy API still works for backward compatibility.

In [14]:
# Method-specific fluent call
semantic_results = await db.query("database").semantic().limit(3).to_list()
print("Semantic API:", len(semantic_results), "results")

# Default fluent query also uses semantic search
query_results = await db.query("database").limit(3).to_list()
print("Default query API:", len(query_results), "results")

Semantic API: 3 results


Default query API: 3 results


## Summary

The Unified API provides:

- **Single entry point**: `db.query("...")`
- **Search modes**: `.semantic()`, `.keyword()`, `.hybrid()`, `.trigram()`
- **Configuration**: `.where()`, `.limit()`, `.ef()`, `.bm25_params()`, `.rrf()`
- **Output formats**: `.to_list()`, `.to_pandas()`, `.to_arrow()`
- **Analysis**: `.explain_plan()`, `.analyze_plan()`

For more details, see the [Migration Guide](../docs/user_guide/migration_guide.md).

In [15]:
# Cleanup
await db.delete_table()
await db.close()
print("🧹 Cleaned up")

🧹 Cleaned up
